# Hands-on 2: from the best run to "production"

We register the best model, give it the alias `champion`, and use it on a stream of alerts, the way Fink does. The MLflow server from Hands-on 1 must still be running.

*On enregistre le meilleur modèle, on lui donne l'alias `champion` et on l'utilise sur un flux d'alertes, comme Fink. Le serveur MLflow doit toujours tourner.*

In [ ]:
import logging
import mlflow
import pandas as pd

logging.getLogger("mlflow").setLevel(logging.ERROR)  # hide MLflow info messages
mlflow.set_tracking_uri("http://127.0.0.1:6969")
client = mlflow.MlflowClient()

## 1. Find the best run

In [ ]:
runs = mlflow.search_runs(experiment_names=["fink-td"], order_by=["metrics.f1 DESC"])
best = runs.iloc[0]
print("best run:", best["tags.mlflow.runName"], "f1 =", round(best["metrics.f1"], 2))

## 2. Register it as `champion`

The **registry** keeps numbered versions of a model. The **alias** `champion` points to the version we want to use.

*Le registre garde les versions du modèle. L'alias `champion` pointe vers la version à utiliser.*

In [ ]:
version = mlflow.register_model(f"runs:/{best['run_id']}/model", "agn-classifier")
client.set_registered_model_alias("agn-classifier", "champion", version.version)

In MLflow, click **Model registry** in the left menu. The first time, a window about the new registry opens: close it with the cross.

*Dans MLflow, cliquez sur **Model registry**. La première fois, une fenêtre s'ouvre : fermez-la avec la croix.*

**You should see / Vous devriez voir :** the model `agn-classifier` in the list. Click on it.

![the model `agn-classifier` in the list. Click on it.](images/07a_registry_list.png)

**You should see / Vous devriez voir :** version 1 with the alias `champion`.

![version 1 with the alias `champion`.](images/07_registry.png)

In Fink, setting this alias on the shared server is what starts the deployment:

```
alias set  ->  webhook  ->  CI  ->  ml-module deployed  ->  the model runs on the live stream
```

Here we skip the server and the CI, and we run the model ourselves.

*Chez Fink, poser cet alias déclenche le déploiement. Ici, on fait tourner le modèle nous-mêmes.*

## 3. Load the champion

In [ ]:
model = mlflow.sklearn.load_model("models:/agn-classifier@champion")

## 4. Run it on a stream of alerts

We send the 320 alerts in batches of 40 and call `processor`, the function Fink calls on each batch.

*On envoie les alertes par paquets de 40 et on appelle `processor`, la fonction que Fink appelle sur chaque paquet.*

In [ ]:
from processor import processor

alerts = pd.read_parquet("data/alerts.parquet")

for start in range(0, len(alerts), 40):
    batch = alerts.iloc[start:start + 40]
    alerts.loc[batch.index, "agn_score"] = processor(batch, model)
    print("batch", start // 40 + 1, ":", (alerts.loc[batch.index, "agn_score"] > 0.5).sum(), "alerts above 0.5")

**You should see / Vous devriez voir :** one line per batch, with the number of alerts that look like an active galaxy.

![one line per batch, with the number of alerts that look like an active galaxy.](images/08_stream.png)

## 5. Write a filter

The module added a column, `agn_score`. A filter uses it. Here: **probably an active galaxy, but unknown to SIMBAD**.

*Le module a ajouté une colonne. Le filtre l'utilise : probablement une galaxie active, mais inconnue de SIMBAD.*

In [ ]:
candidates = alerts[(alerts["agn_score"] > 0.5) & (alerts["cdsxmatch"] == "Unknown")]
candidates[["objectId", "agn_score"]]

**You should see / Vous devriez voir :** a short list of new candidates, with their score.

![a short list of new candidates, with their score.](images/09_candidates.png)

**Exercise:** change 0.5 to 0.8. How many candidates are left? What do we gain, what do we lose?

**Bonus:** train a better model in Hands-on 1, register it, and move the alias to version 2 with `client.set_registered_model_alias("agn-classifier", "champion", 2)`. Re-run sections 3 to 5: the code does not change, only the model does.

*Exercice : changez le seuil. Bonus : déplacez l'alias vers une nouvelle version et relancez, sans toucher au code.*

## In the real Fink / Dans le vrai Fink

| Here | In Fink |
|---|---|
| MLflow on your laptop | the shared Fink MLflow server |
| you set the alias by hand | the alias starts the webhook and the CI |
| a loop over batches of 40 | Spark on the live ZTF and Rubin streams |
| `agn_score` in a notebook | a new column in every alert sent to scientists |